In [1]:
import warnings
warnings.filterwarnings("ignore", message="Attempting to use hipBLASLt")

In [2]:
#%pip install transformers accelerate -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report

### BASELINE: классификатор, построенный на предобученном трансформере (с дообучением).

Т.к. в тестовом наборе важна классификация только объектов, размеченных 0.0 и 1.0, а объекты с разметкой 0.1 (RELEVANT_MINUS) можно игнорировать, то так размеченные объекты были мной удалены и из обучающих данных. 

Задача свелась к бинарной классификации, но первоначально строился классификатор на N-классов. Так всё и осталось, просто N=2. Чуть сложнее конструкция, но на качество классификации это не влияет.

In [3]:
# ============================================================
# ЧАСТЬ 1: Конфигурация
# ============================================================

# --- Воспроизводимость ---
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# --- Устройство ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Устройство: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- Конфигурация модели ---
#MODEL_NAME = "cointegrated/rubert-tiny2"
MODEL_NAME = "DeepPavlov/rubert-base-cased"
MAX_LEN    = 512
#NUM_LABELS = 3
NUM_LABELS = 2

# --- Данные ---
TRAIN_CSV = "./data/df_train_drop_01.csv"
VAL_CSV   = "./data/df_val_drop_01.csv"
TEST_CSV  = "./data/df_test_drop_01.csv"

# --- Маппинги ---
#LABEL_MAP = {0.0: 0, 0.1: 1, 1.0: 2}
#SCORE_MAP = {0: 0.0, 1: 0.1, 2: 1.0}   # обратно: индекс → оригинальный score
#ID2LABEL  = {0: "IRRELEVANT", 1: "RELEVANT_MINUS", 2: "RELEVANT_PLUS"}
LABEL_MAP = {0.0: 0, 1.0: 1}
SCORE_MAP = {0: 0.0, 1: 1.0}   # обратно: индекс → оригинальный score
ID2LABEL  = {0: "IRRELEVANT", 1: "RELEVANT"}

# --- Гиперпараметры (потом можно менять) ---
EPOCHS     = 6
#BATCH_SIZE = 32
BATCH_SIZE = 16
LR         = 2e-6

Устройство: cuda
GPU: AMD Radeon Graphics
VRAM: 17.2 GB


In [4]:
# ============================================================
# ЧАСТЬ 2: Анализ данных
# ============================================================

df_train = pd.read_csv(TRAIN_CSV)
df_val   = pd.read_csv(VAL_CSV)

df = df_train
print(f"Размер датасета: {df.shape}")
print(f"\nКолонки: {df.columns.tolist()}")

# --- Базовая статистика ---
print("\n--- Распределение классов ---")
print(df["relevance"].value_counts())
print()
print(df["relevance"].value_counts(normalize=True).round(3))

# --- Пропуски ---
print("\n--- Пропуски по колонкам ---")
print(df.isnull().sum())

# --- Длины текстов (важно для MAX_LEN) ---
print("\n--- Длины текстовых полей (символы) ---")
for col in ["Text", "normalized_main_rubric_name_ru", "name",
            "address", "prices_summarized", "reviews_summarized"]:
    if col in df.columns:
        lengths = df[col].dropna().str.len()
        print(f"{col:40s}  median={lengths.median():.0f}  max={lengths.max():.0f}")

# --- Несколько примеров по каждому классу ---
print("\n--- Примеры по классам ---")
#for score, label in [(1.0, "RELEVANT_PLUS"), (0.1, "RELEVANT_MINUS"), (0.0, "IRRELEVANT")]:
for score, label in [(1.0, "RELEVANT"), (0.0, "IRRELEVANT")]:
    row = df[df["relevance"] == score].iloc[0]
    print(f"\n[{label}]")
    print(f"  Запрос : {row['Text']}")
    print(f"  Рубрика: {row.get('normalized_main_rubric_name_ru', '—')}")
    print(f"  Название: {row['name']}")
    print(f"  Отзывы : {str(row.get('reviews_summarized', ''))[:120]}...")

Размер датасета: (26585, 8)

Колонки: ['Text', 'address', 'name', 'normalized_main_rubric_name_ru', 'permalink', 'prices_summarized', 'reviews_summarized', 'relevance']

--- Распределение классов ---
relevance
1.0    13998
0.0    12587
Name: count, dtype: int64

relevance
1.0    0.527
0.0    0.473
Name: proportion, dtype: float64

--- Пропуски по колонкам ---
Text                                  0
address                               0
name                                  0
normalized_main_rubric_name_ru        8
permalink                             0
prices_summarized                 10889
reviews_summarized                 1064
relevance                             0
dtype: int64

--- Длины текстовых полей (символы) ---
Text                                      median=25  max=166
normalized_main_rubric_name_ru            median=14  max=57
name                                      median=52  max=3268
address                                   median=38  max=170
prices_summarized   

In [5]:
# ============================================================
# ЧАСТЬ 3: Preprocessing и формирование входа
# ============================================================

# --- Загрузка токенизатора ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Токенизатор загружен: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

Токенизатор загружен: DeepPavlov/rubert-base-cased
Vocab size: 119547


In [6]:
# --- Функция формирования входа ---
def build_input_text(row: pd.Series) -> tuple:
    """
    Возвращает два сегмента:
      segment_a (text)      — поисковый запрос, никогда не обрезается
      segment_b (text_pair) — всё про объект, обрезается с конца

    Порядок полей в segment_b важен:
      рубрика и название — в начало (самый сильный сигнал)
      отзывы — в конец (они и будут обрезаться при overflow)
    """
    query = str(row.get("Text", "")).strip()

    parts = [
        str(row.get("normalized_main_rubric_name_ru", "")),
        str(row.get("name", "")),
        str(row.get("address", "")),
        str(row.get("reviews_summarized", "")),
        str(row.get("prices_summarized", "")),
    ]
    # Убираем пустые строки и "nan"
    parts = [p.strip() for p in parts if p.strip() and p.strip().lower() != "nan"]
    object_text = " | ".join(parts)

    return query, object_text

In [7]:
# --- Проверка на примерах из каждого класса ---
print("\n--- Проверка build_input_text ---")
#for score in [1.0, 0.1, 0.0]:
for score in [1.0, 0.0]:
    row = df[df["relevance"] == score].iloc[0]
    query, object_text = build_input_text(row)
    print(f"\n[relevance={score}]")
    print(f"  segment_a : «{query}»")
    print(f"  segment_b : «{object_text[:120]}...»")


--- Проверка build_input_text ---

[relevance=1.0]
  segment_a : «такелажные компании»
  segment_b : «Строительные леса | Кировская такелажная компания; Kirovskaya Takelazhnaya kompaniya; Ktk43; Такелажная компания; Такела...»

[relevance=0.0]
  segment_a : «Стриптиз клубы ижевск»
  segment_b : «Яхт-клуб | More | Удмуртская Республика, Ижевск, Береговая улица | Общий обзор отзывов: яхт-клуб «More» в Ижевске предос...»


In [8]:
# --- Проверка токенизации одного примера ---
print("\n--- Проверка токенизации ---")
sample_row = df.iloc[0]
query, object_text = build_input_text(sample_row)

encoding = tokenizer(
    text=query,
    text_pair=object_text,
    max_length=MAX_LEN,
    truncation="only_second",
    padding="max_length",
    return_tensors="pt",
)

print(f"input_ids shape     : {encoding['input_ids'].shape}")
print(f"attention_mask shape: {encoding['attention_mask'].shape}")
print(f"token_type_ids shape: {encoding['token_type_ids'].shape}")

# Сколько токенов реально занято (не padding)
real_tokens = encoding["attention_mask"].sum().item()
print(f"Реальных токенов    : {real_tokens} / {MAX_LEN}")

# Декодируем чтобы убедиться что структура правильная
decoded = tokenizer.decode(encoding["input_ids"][0], skip_special_tokens=False)
print(f"\nДекодированный вход:")
print(decoded)


--- Проверка токенизации ---
input_ids shape     : torch.Size([1, 512])
attention_mask shape: torch.Size([1, 512])
token_type_ids shape: torch.Size([1, 512])
Реальных токенов    : 512 / 512

Декодированный вход:
[CLS] Стриптиз клубы ижевск [SEP] Яхт - клуб | More | Удмуртская Республика, Ижевск, Береговая улица | Общий обзор отзывов : яхт - клуб « More » в Ижевске предоставляет услуги по обучению и организации мероприятий на воде. Отзывы исключительно положительные : хвалят атмосферу, профессионализм персонала, качество услуг и удобное расположение. | 1. Хвалят эмоции и кайф от посещения | 2. Отмечают хорошее место | 3. Подчёркивают отличную локацию | 4. Считают место топовым в Ижевске | 5. Рекомендуют для любителей парусного спорта | 6. Отмечают крутость места | 7. Описывают как место силы и отличного настроения | 8. Подчёркивают душевный подход к организации гонок | 9. Советуют посетить клуб | 10. Рекомендуют место | 11. Отмечают красоту мест и отдых | 12. Подчёркивают создание атмо

In [9]:
# --- Статистика по длинам на всём датасете ---
token_lengths = []

for _, row in df.iterrows():
    query, object_text = build_input_text(row)
    encoding = tokenizer(
        text=query,
        text_pair=object_text,
        truncation=False,   # без обрезки — смотрим реальные длины
        padding=False,
    )
    token_lengths.append(len(encoding["input_ids"]))

token_lengths = np.array(token_lengths)
print(f"  median : {np.median(token_lengths):.0f}")
print(f"  90-й перцентиль: {np.percentile(token_lengths, 90):.0f}")
print(f"  95-й перцентиль: {np.percentile(token_lengths, 95):.0f}")
print(f"  max    : {np.max(token_lengths):.0f}")
print(f"  > 512  : {(token_lengths > 512).sum()} строк ({(token_lengths > 512).mean()*100:.1f}%)")

  median : 453
  90-й перцентиль: 598
  95-й перцентиль: 634
  max    : 1540
  > 512  : 8485 строк (31.9%)


In [10]:
# ============================================================
# ЧАСТЬ 4: Dataset класс
# ============================================================

class RelevanceDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_len: int = MAX_LEN,
        has_labels: bool = True,
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        query, object_text = build_input_text(row)

        encoding = self.tokenizer(
            text=query,
            text_pair=object_text,
            max_length=self.max_len,
            truncation="only_second",
            padding="max_length",
            return_tensors="pt",
        )

        item = {
            "input_ids":      encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
        }

        #if "token_type_ids" in encoding:
        #    item["token_type_ids"] = encoding["token_type_ids"].squeeze(0)

        if self.has_labels:
            label = LABEL_MAP[float(row["relevance"])]
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item

In [11]:
# --- Datasets ---
train_dataset = RelevanceDataset(df_train, tokenizer)
val_dataset   = RelevanceDataset(df_val, tokenizer)

# --- Проверка train_dataset ---
print(f"Размер train_dataset: {len(train_dataset)}")

# Один item
item = train_dataset[0]
print(f"\nКлючи item: {list(item.keys())}")
print(f"input_ids shape     : {item['input_ids'].shape}")
print(f"attention_mask shape: {item['attention_mask'].shape}")
print(f"token_type_ids shape: {item['token_type_ids'].shape}")
print(f"label               : {item['labels'].item()} ({ID2LABEL[item['labels'].item()]})")

# --- Проверка по одному примеру каждого класса ---
print("\n--- Проверка по одному примеру каждого класса ---")
#for score, label_name in [(1.0, "RELEVANT_PLUS"), (0.1, "RELEVANT_MINUS"), (0.0, "IRRELEVANT")]:
for score, label_name in [(1.0, "RELEVANT"), (0.0, "IRRELEVANT")]:
    pos = train_dataset.df[train_dataset.df["relevance"] == score].index[0]
    item = train_dataset[pos]
    print(f"  [{label_name}] label={item['labels'].item()}, "
          f"real_tokens={item['attention_mask'].sum().item()}")

Размер train_dataset: 26585

Ключи item: ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
input_ids shape     : torch.Size([512])
attention_mask shape: torch.Size([512])
token_type_ids shape: torch.Size([512])
label               : 0 (IRRELEVANT)

--- Проверка по одному примеру каждого класса ---
  [RELEVANT] label=1, real_tokens=417
  [IRRELEVANT] label=0, real_tokens=512


In [12]:
# ============================================================
# ЧАСТЬ 5: DataLoader-ы, веса классов
# ============================================================
 
# --- DataLoaders ---
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    #num_workers=2,
    #pin_memory=True,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    #num_workers=2,
    #pin_memory=True,
)

print(f"\nТrain батчей: {len(train_loader)}")
print(f"Val батчей  : {len(val_loader)}")

# --- Веса классов для loss ---
counts = df_train["relevance"].map(LABEL_MAP).value_counts().sort_index()
print(f"\nСчетчики классов: {counts}")

class_weights = 1.0 / counts
class_weights = class_weights / class_weights.sum() * NUM_LABELS
class_weights = torch.tensor(class_weights.values, dtype=torch.float32)
print(f"\nВеса классов: {class_weights}")

# --- Быстрая проверка батча ---
print("\n--- Проверка первого батча ---")
batch = next(iter(train_loader))
print(f"input_ids shape     : {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"token_type_ids shape: {batch['token_type_ids'].shape}")
print(f"labels shape        : {batch['labels'].shape}")

unique_values, counts = torch.unique(batch['labels'], return_counts=True)
for val, count in zip(unique_values, counts):
    print(f"Value {val.item()} met {count.item()} times")



Тrain батчей: 1662
Val батчей  : 185

Счетчики классов: relevance
0    12587
1    13998
Name: count, dtype: int64

Веса классов: tensor([1.0531, 0.9469])

--- Проверка первого батча ---
input_ids shape     : torch.Size([16, 512])
attention_mask shape: torch.Size([16, 512])
token_type_ids shape: torch.Size([16, 512])
labels shape        : torch.Size([16])
Value 0 met 7 times
Value 1 met 9 times


In [13]:
# ============================================================
# ЧАСТЬ 6: Модель, loss, оптимизатор, scheduler
# ============================================================

# --- Модель ---
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id={v: k for k, v in ID2LABEL.items()},
    weights_only=False,
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Всего параметров    : {total_params:,}")
print(f"Обучаемых параметров: {trainable_params:,}")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at DeepPavlov/rubert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Всего параметров    : 177,854,978
Обучаемых параметров: 177,854,978


In [14]:
# --- Loss ---
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# --- Оптимизатор ---
# Разные lr для тела трансформера и классификационной головы:
# голова обучается с нуля → ей нужен больший lr
optimizer = AdamW([
    {"params": model.bert.parameters(),        "lr": LR},
    {"params": model.classifier.parameters(),  "lr": LR * 10},
], weight_decay=0.01)

print(f"\nГруппы оптимизатора:")
for i, g in enumerate(optimizer.param_groups):
    print(f"  Группа {i}: lr={g['lr']:.2e}, параметров={sum(p.numel() for p in g['params']):,}")   

# --- Scheduler: linear warmup → linear decay ---
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.05)

print(f"\nШагов всего  : {total_steps}")
print(f"Warmup шагов : {warmup_steps}")

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)


Группы оптимизатора:
  Группа 0: lr=2.00e-06, параметров=177,853,440
  Группа 1: lr=2.00e-05, параметров=1,538

Шагов всего  : 9972
Warmup шагов : 498


In [15]:
# ============================================================
# ЧАСТЬ 7: Цикл обучения
# ============================================================

# --- Функция валидации ---
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["labels"].to(device)

            kwargs = {"input_ids": input_ids, "attention_mask": attention_mask, "token_type_ids": token_type_ids}
            #kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
            #if "token_type_ids" in batch:
            #    kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

            outputs = model(**kwargs)
            loss    = criterion(outputs.logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    return {
        "loss":     total_loss / len(loader),
        "f1_macro": f1_score(all_labels, all_preds, average="macro"),
        "report":   classification_report(
                        all_labels, all_preds,
                        target_names=list(ID2LABEL.values()),
                        digits=4,
                    ),
    }

In [16]:
# --- Цикл обучения ---
best_f1 = 0.0
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad()

        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if "token_type_ids" in batch:
            kwargs["token_type_ids"] = batch["token_type_ids"].to(device)

        outputs = model(**kwargs)
        loss    = criterion(outputs.logits, labels)

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_train_loss = total_train_loss / len(train_loader)

    # --- Валидация ---
    metrics = evaluate(model, val_loader)
    print(f"\nEpoch {epoch}/{EPOCHS}")
    print(f"  train_loss = {avg_train_loss:.4f}")
    print(f"  val_loss   = {metrics['loss']:.4f}")
    print(f"  f1_macro   = {metrics['f1_macro']:.4f}")
    print(metrics["report"])

    # --- Сохраняем лучшую модель ---
    if metrics["f1_macro"] > best_f1:
        best_f1    = metrics["f1_macro"]
        best_epoch = epoch
        model.save_pretrained("best_model")
        #tokenizer.save_pretrained("best_model")
        print(f"  → Лучший F1: {best_f1:.4f}, модель сохранена")

print(f"\nОбучение завершено. Лучший F1-macro: {best_f1:.4f} (epoch {best_epoch})")

Epoch 1/6:   0%|          | 0/1662 [00:00<?, ?it/s]


Epoch 1/6
  train_loss = 0.6105
  val_loss   = 0.5788
  f1_macro   = 0.7003
              precision    recall  f1-score   support

  IRRELEVANT     0.8073    0.5517    0.6555      1488
    RELEVANT     0.6540    0.8655    0.7451      1457

    accuracy                         0.7070      2945
   macro avg     0.7307    0.7086    0.7003      2945
weighted avg     0.7315    0.7070    0.6998      2945

  → Лучший F1: 0.7003, модель сохранена


Epoch 2/6:   0%|          | 0/1662 [00:00<?, ?it/s]


Epoch 2/6
  train_loss = 0.5169
  val_loss   = 0.5209
  f1_macro   = 0.7556
              precision    recall  f1-score   support

  IRRELEVANT     0.7805    0.7191    0.7485      1488
    RELEVANT     0.7344    0.7934    0.7628      1457

    accuracy                         0.7559      2945
   macro avg     0.7574    0.7562    0.7556      2945
weighted avg     0.7577    0.7559    0.7556      2945

  → Лучший F1: 0.7556, модель сохранена


Epoch 3/6:   0%|          | 0/1662 [00:00<?, ?it/s]

...


Epoch 4/6
  train_loss = 0.4450
  val_loss   = 0.5276
  f1_macro   = 0.7624
              precision    recall  f1-score   support

  IRRELEVANT     0.7911    0.7204    0.7541      1488
    RELEVANT     0.7384    0.8058    0.7706      1457

    accuracy                         0.7626      2945
   macro avg     0.7648    0.7631    0.7624      2945
weighted avg     0.7650    0.7626    0.7623      2945

  → Лучший F1: 0.7624, модель сохранена


Epoch 5/6:   0%|          | 0/1662 [00:00<?, ?it/s]


Epoch 5/6
  train_loss = 0.4287
  val_loss   = 0.5299
  f1_macro   = 0.7592
              precision    recall  f1-score   support

  IRRELEVANT     0.7753    0.7372    0.7558      1488
    RELEVANT     0.7444    0.7817    0.7626      1457

    accuracy                         0.7593      2945
   macro avg     0.7599    0.7595    0.7592      2945
weighted avg     0.7600    0.7593    0.7592      2945



Epoch 6/6:   0%|          | 0/1662 [00:00<?, ?it/s]


Epoch 6/6
  train_loss = 0.4154
  val_loss   = 0.5442
  f1_macro   = 0.7594
              precision    recall  f1-score   support

  IRRELEVANT     0.7947    0.7077    0.7487      1488
    RELEVANT     0.7315    0.8133    0.7702      1457

    accuracy                         0.7599      2945
   macro avg     0.7631    0.7605    0.7594      2945
weighted avg     0.7634    0.7599    0.7593      2945


Обучение завершено. Лучший F1-macro: 0.7624 (epoch 4)


In [17]:
df_test = pd.read_csv(TEST_CSV)

In [18]:
print(df_test.shape)
print('\n', df_test.isna().sum())
print('\n', df_test['relevance'].value_counts(dropna=False))

(500, 8)

 Text                                0
address                             0
name                                0
normalized_main_rubric_name_ru      0
permalink                           0
prices_summarized                 207
reviews_summarized                 15
relevance                           0
dtype: int64

 relevance
1.0    317
0.0    183
Name: count, dtype: int64


In [19]:
# --- Проверка ---
print("--- Создаём Dataset из df_test ---")
test_dataset = RelevanceDataset(df_test, tokenizer)
print(f"Размер: {len(test_dataset)}")

# Один item
item = test_dataset[0]
print(f"\nКлючи item: {list(item.keys())}")
print(f"input_ids shape     : {item['input_ids'].shape}")
print(f"attention_mask shape: {item['attention_mask'].shape}")
print(f"token_type_ids shape: {item['token_type_ids'].shape}")
print(f"label               : {item['labels'].item()} ({ID2LABEL[item['labels'].item()]})")

# --- Проверка по одному примеру каждого класса ---
print("\n--- Проверка по одному примеру каждого класса ---")
#for score, label_name in [(1.0, "RELEVANT_PLUS"), (0.1, "RELEVANT_MINUS"), (0.0, "IRRELEVANT")]:
for score, label_name in [(1.0, "RELEVANT"), (0.0, "IRRELEVANT")]:
    pos = test_dataset.df[test_dataset.df["relevance"] == score].index[0]
    item = test_dataset[pos]
    print(f"  [{label_name}] label={item['labels'].item()}, "
          f"real_tokens={item['attention_mask'].sum().item()}")

--- Создаём Dataset из df_test ---
Размер: 500

Ключи item: ['input_ids', 'attention_mask', 'token_type_ids', 'labels']
input_ids shape     : torch.Size([512])
attention_mask shape: torch.Size([512])
token_type_ids shape: torch.Size([512])
label               : 1 (RELEVANT)

--- Проверка по одному примеру каждого класса ---
  [RELEVANT] label=1, real_tokens=451
  [IRRELEVANT] label=0, real_tokens=424


In [20]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    #num_workers=2,
    #pin_memory=True,
)

print(f"\nTest батчей: {len(test_loader)}")

# --- Быстрая проверка батча ---
print("\n--- Проверка первого батча ---")
batch = next(iter(test_loader))
print(f"input_ids shape     : {batch['input_ids'].shape}")
print(f"attention_mask shape: {batch['attention_mask'].shape}")
print(f"token_type_ids shape: {batch['token_type_ids'].shape}")
print(f"labels shape        : {batch['labels'].shape}")

unique_values, counts = torch.unique(batch['labels'], return_counts=True)
for val, count in zip(unique_values, counts):
    print(f"Value {val.item()} met {count.item()} times")


Test батчей: 32

--- Проверка первого батча ---
input_ids shape     : torch.Size([16, 512])
attention_mask shape: torch.Size([16, 512])
token_type_ids shape: torch.Size([16, 512])
labels shape        : torch.Size([16])
Value 0 met 6 times
Value 1 met 10 times


In [21]:
# Загрузка лучшей модели
#tokenizer = AutoTokenizer.from_pretrained("best_model")
model = AutoModelForSequenceClassification.from_pretrained(
    "best_model",
    weights_only=False,
)
model = model.to(device)
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(119547, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1

In [22]:
# --- Валидация (val) ---
metrics = evaluate(model, val_loader)
print(f"  val_loss   = {metrics['loss']:.4f}")
print(f"  f1_macro   = {metrics['f1_macro']:.4f}")
print(metrics["report"])

  val_loss   = 0.5276
  f1_macro   = 0.7624
              precision    recall  f1-score   support

  IRRELEVANT     0.7911    0.7204    0.7541      1488
    RELEVANT     0.7384    0.8058    0.7706      1457

    accuracy                         0.7626      2945
   macro avg     0.7648    0.7631    0.7624      2945
weighted avg     0.7650    0.7626    0.7623      2945



In [23]:
# --- Валидация (test) ---
metrics = evaluate(model, test_loader)
print(f"  test_loss   = {metrics['loss']:.4f}")
print(f"  f1_macro   = {metrics['f1_macro']:.4f}")
print(metrics["report"])

  test_loss   = 0.6275
  f1_macro   = 0.7208
              precision    recall  f1-score   support

  IRRELEVANT     0.6129    0.7268    0.6650       183
    RELEVANT     0.8233    0.7350    0.7767       317

    accuracy                         0.7320       500
   macro avg     0.7181    0.7309    0.7208       500
weighted avg     0.7463    0.7320    0.7358       500



#### ВЫВОДЫ:

Потребовалось много попыток подбора гиперпараметров модели: LR для тела трансформера, LR для головы трансформера, значения WEGHT_DECAY, продолжительности WARMUP и количества эпох обучения.

В результате удалось повысить качество на тестовых данных с первоначальных около 50% до 73,20%. Причем, качество классификации модели резко повысилось после отказа от промежуточного третьего класса 0.1 (RELEVANT_MINUS).

Т.о. baseline, с которым будем сравнивать качество работы агента на тестовых данных: 73,20%.